# Data quality pass — `application_train`

Full data-quality audit of the model feature table, run **before any modeling or imputation decision**. Scope is `application_train` only: this is the table that carries `TARGET` and will become the model matrix, so it is the table whose feature distributions actually need auditing. `bureau` and `previous_application` are relational history tables that get aggregated into `application_train`-level features later — auditing them column-by-column here wouldn't tell us anything about model-input quality yet.

For every feature we report:

- **numeric (continuous) features** — % missing, skewness, and outliers flagged by the IQR method (values outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`)
- **categorical features** — cardinality (distinct value count) and % missing

**Column classification rule.** `application_train` mixes true continuous numerics (income, credit amount, EXT_SOURCE scores) with integer columns that are actually two-valued indicators (`FLAG_EMAIL`, `FLAG_DOCUMENT_3`, ...). Skewness and IQR fences are meaningless for a 0/1 variable, so any numeric column with **2 or fewer distinct non-null values** is reclassified as categorical (cardinality ≤ 2) rather than continuous. Everything else numeric (`BIGINT`/`DOUBLE` with >2 distinct values) is treated as continuous. String and boolean columns are categorical by construction. `SK_ID_CURR` (identifier) and `TARGET` (label) are excluded from both tables — they aren't features.

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', 150)
pd.set_option('display.width', 140)

con = duckdb.connect('../data/processed/creditbridge.duckdb')
total = con.execute('SELECT COUNT(*) FROM application_train').fetchone()[0]
print(f'application_train: {total:,} rows')

application_train: 307,511 rows


In [2]:
desc = con.execute('DESCRIBE application_train').fetchdf()
numeric_sql_types = {'BIGINT', 'DOUBLE', 'INTEGER'}
id_cols = {'SK_ID_CURR', 'TARGET'}

feature_cols = desc[~desc.column_name.isin(id_cols)].copy()
numeric_candidates = feature_cols[feature_cols.column_type.isin(numeric_sql_types)].column_name.tolist()

nunique = {
    c: con.execute(f'SELECT COUNT(DISTINCT "{c}") FROM application_train').fetchone()[0]
    for c in numeric_candidates
}

continuous_numeric = [c for c in numeric_candidates if nunique[c] > 2]
binary_flags = [c for c in numeric_candidates if nunique[c] <= 2]
categorical_cols = feature_cols[~feature_cols.column_type.isin(numeric_sql_types)].column_name.tolist()

print(f'{len(continuous_numeric)} continuous numeric features')
print(f'{len(binary_flags)} binary 0/1 flag columns (reclassified as categorical)')
print(f'{len(categorical_cols)} string/boolean categorical features')
print(f'{len(continuous_numeric) + len(binary_flags) + len(categorical_cols)} total features audited (of {len(feature_cols)} non-id/target columns)')

72 continuous numeric features
32 binary 0/1 flag columns (reclassified as categorical)
16 string/boolean categorical features
120 total features audited (of 120 non-id/target columns)


## Numeric features: % missing, skewness, IQR outliers

`skewness()` and `quantile_cont()` are DuckDB's native aggregate functions — computed in-engine rather than pulling 307k rows into pandas per column, which keeps this pass fast even across 72 continuous features. IQR fences use the standard `1.5 * IQR` multiplier.

In [3]:
rows = []
for c in continuous_numeric:
    missing, skew, q1, q3 = con.execute(f'''
        SELECT
            SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END),
            skewness("{c}"),
            quantile_cont("{c}", 0.25),
            quantile_cont("{c}", 0.75)
        FROM application_train
    ''').fetchone()

    if q1 is None or q3 is None:
        iqr_lo = iqr_hi = n_outliers = pct_outliers = None
    else:
        iqr = q3 - q1
        iqr_lo, iqr_hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_outliers = con.execute(f'''
            SELECT COUNT(*) FROM application_train
            WHERE "{c}" IS NOT NULL AND ("{c}" < {iqr_lo} OR "{c}" > {iqr_hi})
        ''').fetchone()[0]
        pct_outliers = round(100 * n_outliers / total, 2)

    rows.append({
        'feature': c,
        'pct_missing': round(100 * missing / total, 2),
        'skewness': round(skew, 2) if skew is not None else None,
        'iqr_lower_fence': iqr_lo,
        'iqr_upper_fence': iqr_hi,
        'n_outliers_iqr': n_outliers,
        'pct_outliers_iqr': pct_outliers,
    })

numeric_dq = pd.DataFrame(rows).sort_values('pct_missing', ascending=False).reset_index(drop=True)
numeric_dq

,feature,pct_missing,skewness,iqr_lower_fence,iqr_upper_fence,n_outliers_iqr,pct_outliers_iqr
0,COMMONAREA_MODE,69.87,5.62,-0.055500,1.117000e-01,7938,2.58
1,COMMONAREA_MEDI,69.87,5.42,-0.057200,1.164000e-01,7992,2.60
2,COMMONAREA_AVG,69.87,5.46,-0.057750,1.170500e-01,7942,2.58
3,NONLIVINGAPARTMENTS_AVG,69.43,15.54,-0.005850,9.750000e-03,15580,5.07
4,NONLIVINGAPARTMENTS_MEDI,69.43,15.67,-0.005850,9.750000e-03,15215,4.95
5,NONLIVINGAPARTMENTS_MODE,69.43,16.25,-0.005850,9.750000e-03,14224,4.63
6,LIVINGAPARTMENTS_AVG,68.35,3.04,-0.055500,2.269000e-01,7881,2.56
7,LIVINGAPARTMENTS_MODE,68.35,2.90,-0.061450,2.469500e-01,7469,2.43
8,LIVINGAPARTMENTS_MEDI,68.35,2.99,-0.056400,2.308000e-01,7927,2.58
9,FLOORSMIN_AVG,67.85,0.95,-0.354250,8.125500e-01,343,0.11


## Categorical features (incl. binary flags): cardinality, % missing

In [4]:
cat_all = categorical_cols + binary_flags
rows = []
for c in cat_all:
    missing, card = con.execute(f'''
        SELECT SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END), COUNT(DISTINCT "{c}")
        FROM application_train
    ''').fetchone()
    rows.append({
        'feature': c,
        'cardinality': card,
        'pct_missing': round(100 * missing / total, 2),
        'kind': 'binary_flag' if c in binary_flags else 'categorical',
    })

categorical_dq = pd.DataFrame(rows).sort_values('pct_missing', ascending=False).reset_index(drop=True)
categorical_dq

,feature,cardinality,pct_missing,kind
0,FONDKAPREMONT_MODE,4,68.39,categorical
1,WALLSMATERIAL_MODE,7,50.84,categorical
2,HOUSETYPE_MODE,3,50.18,categorical
3,EMERGENCYSTATE_MODE,2,47.40,categorical
4,OCCUPATION_TYPE,18,31.35,categorical
5,NAME_TYPE_SUITE,7,0.42,categorical
6,FLAG_OWN_CAR,2,0.00,categorical
7,NAME_CONTRACT_TYPE,2,0.00,categorical
8,NAME_FAMILY_STATUS,6,0.00,categorical
9,NAME_EDUCATION_TYPE,5,0.00,categorical


## Notable findings feeding into `src/clean.py`

1. **EXT_SOURCE_1** is missing on ~56% of applicants, **EXT_SOURCE_3** on ~19%, **EXT_SOURCE_2** on ~0.2%. These are external bureau-style credit scores and the single strongest predictors in this dataset — their *absence* is itself informative (it correlates with thin-file status), which is why they get different treatment than other numeric columns below.
2. Housing/building descriptors (`*_AVG`, `*_MODE`, `*_MEDI` — apartment size, number of elevators, wall material, etc.) are missing on roughly half the book. These describe the applicant's building, populated only when a property is registered as collateral — missingness reflects loan type, not data capture failure.
3. **`DAYS_EMPLOYED` contains a sentinel value, not a missing-value problem.** The column has 0% nulls, but check the actual max:

In [5]:
sentinel_count = con.execute('SELECT COUNT(*) FROM application_train WHERE DAYS_EMPLOYED = 365243').fetchone()[0]
print(f"DAYS_EMPLOYED == 365243 (~1,000 years): {sentinel_count:,} rows ({100*sentinel_count/total:.1f}% of applicants)")
print()
print(numeric_dq[numeric_dq.feature == 'DAYS_EMPLOYED'].to_string(index=False))

DAYS_EMPLOYED == 365243 (~1,000 years): 55,374 rows (18.0% of applicants)

      feature  pct_missing  skewness  iqr_lower_fence  iqr_upper_fence  n_outliers_iqr  pct_outliers_iqr
DAYS_EMPLOYED          0.0      1.66          -6466.5           3417.5           72217             23.48


5. **The IQR method breaks down on near-discrete/zero-inflated columns — 9 of them.** `1.5 * IQR` fences assume a reasonably continuous distribution around the middle 50%. When a column's 25th and 75th percentile are the *same* value (`IQR == 0`), the fence collapses to a single point and *any* different value gets flagged as an "outlier" — including the majority of a legitimate, lumpy distribution. Check which continuous-numeric columns have `IQR == 0`:

In [6]:
zero_iqr_cols = numeric_dq[numeric_dq.iqr_lower_fence == numeric_dq.iqr_upper_fence].feature.tolist()
print(f'{len(zero_iqr_cols)} columns have IQR == 0:')
for c in zero_iqr_cols:
    top_val, top_n = con.execute(f'''
        SELECT "{c}", COUNT(*) n FROM application_train
        GROUP BY 1 ORDER BY n DESC LIMIT 1
    ''').fetchone()
    pct = 100 * top_n / total
    print(f'  {c:<28} {pct:5.1f}% of rows equal {top_val}')

9 columns have IQR == 0:
  AMT_REQ_CREDIT_BUREAU_HOUR    86.0% of rows equal 0.0
  AMT_REQ_CREDIT_BUREAU_MON     72.3% of rows equal 0.0
  AMT_REQ_CREDIT_BUREAU_QRT     70.1% of rows equal 0.0
  AMT_REQ_CREDIT_BUREAU_WEEK    83.7% of rows equal 0.0
  AMT_REQ_CREDIT_BUREAU_DAY     86.0% of rows equal 0.0
  DEF_30_CNT_SOCIAL_CIRCLE      88.2% of rows equal 0.0
  DEF_60_CNT_SOCIAL_CIRCLE      91.3% of rows equal 0.0
  REGION_RATING_CLIENT_W_CITY   74.6% of rows equal 2
  REGION_RATING_CLIENT          73.8% of rows equal 2


Two distinct shapes hide behind `IQR == 0`:

- **`REGION_RATING_CLIENT` / `REGION_RATING_CLIENT_W_CITY`** — a 3-level ordinal rating (1/2/3), not a continuous measurement. ~74% of applicants sit at the modal rating, so `Q1 == Q3 == 2` and the "outlier" flag (~26% of rows) is a pure artifact of applying a continuous-distribution test to an ordinal variable with only 3 possible values.
- **`DEF_30/60_CNT_SOCIAL_CIRCLE`, `AMT_REQ_CREDIT_BUREAU_HOUR/DAY/WEEK/MON/QRT`** — zero-inflated counts (defaults in an applicant's social circle; credit bureau inquiries in a given window). The overwhelming majority of applicants are legitimately at 0, so `Q1 == Q3 == 0` and every nonzero count gets flagged, even though a handful of recent inquiries is normal, observed behavior, not a data error.

In both cases the *column*, not just a few rows, is degenerate under the 1.5×IQR rule. `src/clean.py` treats `IQR == 0` as "the IQR test isn't meaningful for this column" and skips percentile-capping there — capping is applied only where the IQR method actually distinguishes a normal range from a genuine tail. See the "outlier handling" section of `reports/segment_findings.md` for the full rationale.

365,243 days is the dataset's placeholder for "not currently employed" (mostly pensioners) — not a missing value, a mis-encoded one. It shows up as an extreme outlier by construction: with real employment tenures maxing out around 49 years (~17,900 days), a value of 1,000 years sits enormously past the IQR fence, which is exactly what the pass above catches (`n_outliers_iqr` = 72,217 for `DAYS_EMPLOYED`).

**Correction — percentile capping alone does *not* fix this.** The initial version of this notebook claimed 1st/99th-percentile capping would neutralize the sentinel as a side effect of the general outlier rule. Running `src/clean.py` against the real data disproved that: the sentinel is common enough (18% of applicants) that it *defines* the column's own 99th percentile — capping to `p99` caps to the sentinel value itself, leaving all 55,374 rows untouched at 365,243. Generic percentile capping only works when the anomaly is a minority of the tail; here it's the majority of the tail.

The actual fix in `src/clean.py`: `DAYS_EMPLOYED == 365243` is treated as a proper sentinel — converted to `NULL` *before* any statistics are computed, with a `DAYS_EMPLOYED_IS_RETIRED` flag capturing the fact separately (the same "don't silently erase an informative signal" logic used for `EXT_SOURCE`). The remaining genuine tenure values then go through the normal median-impute + IQR-capping path untouched by the sentinel. See `reports/segment_findings.md` for the full writeup — this is the kind of finding that only surfaces by actually running the cleaning code and checking the result, not by reasoning about it in the abstract.

4. Monetary features (`AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`) are all strongly right-skewed (skewness far above 1), as expected for income/credit-amount data — a small number of high earners/large loans stretch the upper tail. This motivates percentile capping over dropping outlier rows: the outliers are real applicants, not data errors, and dropping them would bias the sample toward lower incomes.